# FINOVA · VAN con Financiamiento — EC-300

Este notebook genera una **aplicación HTML autónoma e interactiva** para evaluar proyectos con financiamiento bancario mediante VAN.

## Lógica académica implementada
- Ingresos y gastos proyectados por año.
- Inversión inicial y tasa de descuento/costo de fondos.
- Financiamiento mediante **cuota nivelada (sistema francés)**, con periodicidad configurable.
- Separación anual de la cuota en **interés** y **capital**.
- Método principal usado en clase: `Flujo ajustado = Flujo neto − Cuota + Capital = Flujo neto − Interés`.
- VAN, TIR del flujo ajustado, payback, tabla de amortización y sensibilidad.
- Motor numérico que calcula ingresos/gastos/tasa bancaria/inversión de equilibrio.
- **Modo exposición** con introducción, datos, financiamiento, resultado y conclusiones.

El caso cargado por defecto reproduce el ejemplo de **₡5.600.000, 60% financiado, 14% nominal anual, 4 años y tasa de descuento del 20%**.


## 1. Auditoría matemática del ejemplo base
Esta celda demuestra que el cálculo de la aplicación reproduce la estructura del ejercicio y permite revisar la matemática antes de presentar.


In [1]:
from math import pow

# Caso base visto en clase
investment = 5_600_000
financing_pct = 0.60
loan_rate_nominal = 0.14
payments_per_year = 12
loan_years = 4
discount_rate = 0.20

incomes = [3_744_000, 4_212_000, 3_510_000, 2_340_000]
expenses = [672_000, 1_572_000, 1_022_000, 772_000]

loan = investment * financing_pct
period_rate = loan_rate_nominal / payments_per_year
n_payments = loan_years * payments_per_year
payment = loan * period_rate / (1 - (1 + period_rate) ** (-n_payments))

balance = loan
annual = []
for year in range(loan_years):
    interest_y = principal_y = payment_y = 0.0
    for _ in range(payments_per_year):
        interest = balance * period_rate
        principal = payment - interest
        balance -= principal
        interest_y += interest
        principal_y += principal
        payment_y += payment
    annual.append((payment_y, interest_y, principal_y))

net_flows = [i - g for i, g in zip(incomes, expenses)]
adjusted_flows = [
    net_flows[t] - annual[t][0] + annual[t][2]  # flujo neto - cuota + capital
    for t in range(4)
]
present_values = [
    adjusted_flows[t] / (1 + discount_rate) ** (t + 1)
    for t in range(4)
]
van = -investment + sum(present_values)

print(f"Monto financiado: {loan:,.2f}")
print(f"Cuota mensual nivelada: {payment:,.2f}")
print(f"Servicio anual de deuda: {annual[0][0]:,.2f}")
print("Capital amortizado por año:", [round(x[2]) for x in annual])
print("Interés por año:", [round(x[1]) for x in annual])
print("Flujo ajustado:", [round(x) for x in adjusted_flows])
print(f"VAN: {van:,.2f}")
print("La diferencia de 1–2 unidades respecto a algunas celdas de Excel se debe al redondeo intermedio.")


Monto financiado: 3,360,000.00
Cuota mensual nivelada: 91,816.96
Servicio anual de deuda: 1,101,803.53
Capital amortizado por año: [673536, 774124, 889733, 1022607]
Interés por año: [428267, 327680, 212071, 79196]
Flujo ajustado: [2643733, 2312320, 2275929, 1488804]
VAN: 243,957.38
La diferencia de 1–2 unidades respecto a algunas celdas de Excel se debe al redondeo intermedio.


## 2. Generar la aplicación
Ejecute la siguiente celda. Se creará `FINOVA_VAN_Financiamiento_EC300.html` en la misma carpeta del notebook. El archivo HTML **no necesita servidor ni librerías externas** y puede abrirse directamente en un navegador.


In [2]:
from pathlib import Path
from IPython.display import IFrame, display, Markdown

HTML_APP = r"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>FINOVA · VAN con Financiamiento · EC-300</title>
<style>
:root{
  --navy:#0b1f5b; --blue:#153b9b; --cyan:#16b9d4; --yellow:#ffd400;
  --bg:#f3f6fb; --card:#ffffff; --text:#16213d; --muted:#6d7890;
  --line:#dfe6f2; --green:#0e9f6e; --red:#d14343; --orange:#d97706;
  --shadow:0 18px 50px rgba(16,35,78,.10);
  --radius:22px;
}
*{box-sizing:border-box}
html{scroll-behavior:smooth}
body{margin:0;font-family:Inter,ui-sans-serif,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif;background:var(--bg);color:var(--text)}
button,input,select{font:inherit}
button{cursor:pointer}
.app{min-height:100vh;display:grid;grid-template-columns:258px 1fr}
.sidebar{
  background:linear-gradient(180deg,#071747 0%,#0d2f82 76%,#0a4d8e 100%);
  color:#fff;padding:24px 18px;position:sticky;top:0;height:100vh;overflow:auto
}
.brand{display:flex;align-items:center;gap:12px;padding:8px 10px 24px}
.logo{
  width:44px;height:44px;border-radius:14px;display:grid;place-items:center;font-weight:900;font-size:20px;
  background:linear-gradient(135deg,var(--cyan),#6be3f4);color:#071747;box-shadow:0 8px 25px rgba(22,185,212,.25)
}
.brand b{font-size:19px;letter-spacing:.5px}.brand small{display:block;color:#b8c9ef;margin-top:3px}
.nav{display:grid;gap:6px}.nav button{
  border:0;background:transparent;color:#dce6ff;text-align:left;padding:12px 13px;border-radius:12px;
  display:flex;gap:10px;align-items:center
}
.nav button:hover,.nav button.active{background:rgba(255,255,255,.12);color:#fff}
.nav .dot{width:8px;height:8px;border-radius:50%;background:#6e8fd8}.nav button.active .dot{background:var(--yellow)}
.side-note{margin:26px 9px 0;padding:14px;border:1px solid rgba(255,255,255,.15);border-radius:15px;color:#c9d6f6;font-size:12px;line-height:1.5}
.main{min-width:0}
.topbar{
  height:76px;background:rgba(255,255,255,.92);backdrop-filter:blur(12px);
  border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;padding:0 28px;
  position:sticky;top:0;z-index:20
}
.top-title{min-width:0}.top-title small{display:block;color:var(--muted);font-weight:700;letter-spacing:.04em;text-transform:uppercase;font-size:11px}
.top-title strong{font-size:18px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;display:block;max-width:600px}
.actions{display:flex;gap:9px;align-items:center}
.btn{border:1px solid var(--line);background:#fff;color:var(--text);padding:10px 13px;border-radius:11px;font-weight:750;box-shadow:0 2px 8px rgba(0,0,0,.02)}
.btn:hover{transform:translateY(-1px)}
.btn.primary{background:var(--blue);border-color:var(--blue);color:#fff}
.btn.cyan{background:#e9fbfe;border-color:#b8f0f6;color:#076778}
.content{padding:28px;max-width:1500px;margin:auto}
.section{display:none}.section.active{display:block}
.hero{
  background:radial-gradient(circle at 86% 10%,rgba(22,185,212,.22),transparent 28%),
  linear-gradient(135deg,#071747,#153b9b 72%,#1556b7);
  color:#fff;border-radius:28px;padding:30px;box-shadow:var(--shadow);position:relative;overflow:hidden
}
.hero:after{content:"";position:absolute;width:250px;height:250px;border:42px solid rgba(255,212,0,.08);border-radius:50%;right:-90px;bottom:-150px}
.hero-grid{display:grid;grid-template-columns:1.35fr .65fr;gap:22px;position:relative;z-index:1}
.eyebrow{font-size:12px;font-weight:900;letter-spacing:.12em;text-transform:uppercase;color:#8feafa}
.hero h1{font-size:38px;line-height:1.05;margin:10px 0 12px;max-width:800px}
.hero p{color:#dce8ff;max-width:800px;line-height:1.6;margin:0}
.decision-panel{background:rgba(255,255,255,.10);border:1px solid rgba(255,255,255,.20);border-radius:20px;padding:20px;align-self:stretch;display:flex;flex-direction:column;justify-content:center}
.decision-label{font-size:12px;color:#c6d6f5;text-transform:uppercase;font-weight:900;letter-spacing:.1em}
.decision-main{font-size:30px;font-weight:950;margin:7px 0}
.decision-sub{font-size:13px;color:#dce8ff;line-height:1.45}
.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:14px;margin-top:18px}
.kpi{background:var(--card);border:1px solid var(--line);border-radius:18px;padding:18px;box-shadow:0 8px 24px rgba(17,40,86,.05)}
.kpi small{color:var(--muted);font-weight:800}.kpi .val{font-size:25px;font-weight:950;margin-top:8px}.kpi .sub{font-size:12px;color:var(--muted);margin-top:4px}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:18px;margin-top:18px}
.grid3{display:grid;grid-template-columns:repeat(3,1fr);gap:18px;margin-top:18px}
.card{background:var(--card);border:1px solid var(--line);border-radius:var(--radius);padding:22px;box-shadow:0 8px 25px rgba(16,35,78,.055)}
.card h2,.card h3{margin:0 0 7px}.card p.lead{color:var(--muted);margin:0 0 18px;line-height:1.55}
.section-title{display:flex;align-items:flex-end;justify-content:space-between;gap:16px;margin-bottom:18px}
.section-title h1{margin:0;font-size:28px}.section-title p{margin:5px 0 0;color:var(--muted)}
.form-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}
.field label{display:block;font-size:12px;color:var(--muted);font-weight:850;margin:0 0 6px}
.field input,.field select{
  width:100%;border:1px solid #ced8e8;border-radius:11px;padding:11px 12px;background:#fbfcff;outline:none;color:var(--text)
}
.field input:focus,.field select:focus{border-color:var(--cyan);box-shadow:0 0 0 3px rgba(22,185,212,.11)}
.inline-range{display:grid;grid-template-columns:1fr 92px;gap:9px;align-items:center}.inline-range input[type=range]{padding:0;border:0;box-shadow:none}
.table-wrap{overflow:auto;border:1px solid var(--line);border-radius:15px}
table{border-collapse:collapse;width:100%;font-size:13px}
th{background:#f5f8fd;color:#53617b;text-align:right;font-size:11px;text-transform:uppercase;letter-spacing:.05em;padding:11px 10px;border-bottom:1px solid var(--line);white-space:nowrap}
th:first-child,td:first-child{text-align:left}
td{padding:11px 10px;border-bottom:1px solid #edf1f7;text-align:right;white-space:nowrap}
tr:last-child td{border-bottom:0}.row-input{width:128px;padding:8px 9px;border:1px solid #d8e1ef;border-radius:9px;text-align:right}
.badge{display:inline-flex;align-items:center;gap:7px;border-radius:999px;padding:7px 10px;font-size:12px;font-weight:900}
.badge.green{background:#e8f8f2;color:#087350}.badge.red{background:#fff0f0;color:#ad2c2c}.badge.orange{background:#fff5e7;color:#9a5605}.badge.blue{background:#eef3ff;color:#2b4f9e}
.mini-stat{padding:15px;border-radius:15px;background:#f7f9fd;border:1px solid #e7edf6}.mini-stat small{color:var(--muted);font-weight:800}.mini-stat strong{display:block;font-size:20px;margin-top:6px}
.callout{padding:15px 16px;border-radius:14px;background:#eefafe;border-left:4px solid var(--cyan);line-height:1.55;color:#32415f}
.callout.warn{background:#fff8eb;border-left-color:var(--orange)}.callout.good{background:#edf9f4;border-left-color:var(--green)}.callout.bad{background:#fff1f1;border-left-color:var(--red)}
.formula{background:#091942;color:#e8f1ff;border-radius:14px;padding:15px 17px;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;line-height:1.7;overflow:auto;font-size:13px}
.finance-ring{width:170px;height:170px;border-radius:50%;margin:2px auto 14px;display:grid;place-items:center;position:relative;background:conic-gradient(var(--cyan) 0 60%,#dce5f2 60% 100%)}
.finance-ring:after{content:"";position:absolute;width:116px;height:116px;background:#fff;border-radius:50%}
.finance-ring div{z-index:1;text-align:center}.finance-ring strong{display:block;font-size:24px}.finance-ring small{color:var(--muted)}
.bar-chart{display:flex;align-items:flex-end;gap:14px;height:230px;padding:18px 12px 35px;border-left:1px solid #e1e7f0;border-bottom:1px solid #e1e7f0;position:relative}
.bar-col{flex:1;display:flex;gap:4px;align-items:flex-end;justify-content:center;height:100%;position:relative;min-width:55px}.bar{width:34%;min-width:12px;border-radius:8px 8px 2px 2px;position:relative}.bar.flow{background:linear-gradient(180deg,#16b9d4,#0e85a2)}.bar.pv{background:linear-gradient(180deg,#2757b9,#142f7c)}.bar-col label{position:absolute;bottom:-27px;font-size:11px;color:var(--muted);font-weight:800}
.legend{display:flex;gap:18px;font-size:12px;color:var(--muted);margin-top:10px}.legend i{display:inline-block;width:10px;height:10px;border-radius:3px;margin-right:5px}
.sens-grid{display:grid;grid-template-columns:80px repeat(5,1fr);gap:5px;min-width:600px}.sens-cell{padding:12px 8px;text-align:center;border-radius:9px;font-size:12px;font-weight:850}.sens-head{background:#eef2f8;color:#687590}.sens-pos{background:#e7f7ef;color:#0b7551}.sens-neg{background:#ffeaea;color:#b03030}.sens-mid{background:#fff5df;color:#8f5a09}
.recommendations{display:grid;gap:10px}.rec{display:grid;grid-template-columns:34px 1fr;gap:10px;align-items:flex-start;padding:13px;border:1px solid var(--line);border-radius:13px}.rec .num{width:30px;height:30px;border-radius:9px;background:#edf3ff;color:var(--blue);display:grid;place-items:center;font-weight:950}.rec p{margin:0;line-height:1.55;color:#42516d}
.presentation{background:#061644;color:#fff;min-height:680px;border-radius:26px;overflow:hidden;box-shadow:var(--shadow);position:relative}
.slide{display:none;min-height:680px;padding:58px}.slide.active{display:flex;flex-direction:column;justify-content:center}
.slide .kicker{color:#7be2f2;font-weight:900;text-transform:uppercase;letter-spacing:.13em;font-size:12px}.slide h2{font-size:46px;line-height:1.05;margin:12px 0 18px;max-width:950px}.slide p,.slide li{font-size:20px;line-height:1.55;color:#d9e7ff;max-width:1000px}.slide ul{padding-left:22px}.slide .big{font-size:62px;font-weight:950;color:#fff;margin:10px 0}.slide .accent{color:var(--yellow)}
.slide-controls{position:absolute;bottom:24px;left:24px;right:24px;display:flex;justify-content:space-between;align-items:center}.slide-controls button{border:1px solid rgba(255,255,255,.25);background:rgba(255,255,255,.08);color:#fff;padding:10px 14px;border-radius:11px}.slide-counter{color:#a8bce7;font-size:13px}
.warning-strip{display:none;margin-top:14px;padding:12px 14px;border-radius:12px;background:#fff3df;color:#8a5708;border:1px solid #f4d7a1;font-size:13px;line-height:1.45}
.footer{color:#7b879d;font-size:11px;margin:26px 4px 8px;line-height:1.55}
input[type=range]{accent-color:var(--blue)}
@media(max-width:1050px){.app{grid-template-columns:1fr}.sidebar{position:relative;height:auto}.nav{grid-template-columns:repeat(3,1fr)}.side-note{display:none}.topbar{top:0}.hero-grid,.grid2,.grid3{grid-template-columns:1fr}.kpis{grid-template-columns:repeat(2,1fr)}}
@media(max-width:680px){.content{padding:15px}.topbar{padding:0 14px}.actions .hide-mobile{display:none}.form-grid{grid-template-columns:1fr}.kpis{grid-template-columns:1fr}.hero h1{font-size:30px}.nav{grid-template-columns:1fr 1fr}.slide{padding:34px 24px}.slide h2{font-size:34px}.slide p,.slide li{font-size:17px}}
@media print{.sidebar,.topbar,.actions,.slide-controls{display:none!important}.app{display:block}.content{padding:0}.section{display:block!important;page-break-after:always}.presentation{min-height:auto}.slide{display:block!important;min-height:0;page-break-after:always;color:#111;background:#fff}.slide p,.slide li{color:#222}.slide h2,.slide .big{color:#111}.slide .accent{color:#8a6900}}
</style>
</head>
<body>
<div class="app">
  <aside class="sidebar">
    <div class="brand"><div class="logo">FV</div><div><b>FINOVA</b><small>VAN con Financiamiento</small></div></div>
    <nav class="nav">
      <button class="active" data-section="dashboard"><span class="dot"></span>Decisión</button>
      <button data-section="proyecto"><span class="dot"></span>Proyecto</button>
      <button data-section="credito"><span class="dot"></span>Financiamiento</button>
      <button data-section="van"><span class="dot"></span>Cálculo VAN</button>
      <button data-section="sensibilidad"><span class="dot"></span>Sensibilidad</button>
      <button data-section="exposicion"><span class="dot"></span>Modo exposición</button>
    </nav>
    <div class="side-note">
      <b>EC-300 · Modelo académico</b><br>
      Replica el procedimiento del ejemplo de clase y agrega diagnóstico numérico, sensibilidad y presentación.
    </div>
  </aside>

  <main class="main">
    <header class="topbar">
      <div class="top-title"><small>Herramientas Tecnológicas Aplicadas · EC-300</small><strong id="topProject">VAN con financiamiento</strong></div>
      <div class="actions">
        <button class="btn cyan hide-mobile" id="btnExample">↺ Ejemplo del profesor</button>
        <button class="btn hide-mobile" id="btnExport">⇩ Exportar escenario</button>
        <button class="btn primary" onclick="window.print()">Imprimir / PDF</button>
      </div>
    </header>

    <div class="content">
      <section class="section active" id="dashboard">
        <div class="hero">
          <div class="hero-grid">
            <div>
              <div class="eyebrow">Motor de decisión financiera</div>
              <h1 id="heroProject">Proyecto demostrativo · VAN con financiamiento</h1>
              <p>Evalúa inversión, deuda bancaria, flujos, costo de fondos y valor del dinero en el tiempo. El resultado principal sigue el esquema usado en clase: <b>flujo neto − cuota + capital</b>, que equivale a descontar el flujo operativo menos el componente de interés.</p>
            </div>
            <div class="decision-panel">
              <div class="decision-label">Decisión VAN</div>
              <div class="decision-main" id="decisionMain">—</div>
              <div class="decision-sub" id="decisionSub">Complete los datos para evaluar el proyecto.</div>
            </div>
          </div>
        </div>

        <div class="kpis">
          <div class="kpi"><small>VAN con financiamiento</small><div class="val" id="kpiNPV">—</div><div class="sub" id="kpiNPVSub">Valor presente neto</div></div>
          <div class="kpi"><small>TIR del flujo ajustado</small><div class="val" id="kpiIRR">—</div><div class="sub" id="kpiIRRSub">vs. tasa requerida</div></div>
          <div class="kpi"><small>Financiamiento bancario</small><div class="val" id="kpiDebt">—</div><div class="sub" id="kpiDebtSub">porcentaje financiado</div></div>
          <div class="kpi"><small>Payback descontado</small><div class="val" id="kpiPayback">—</div><div class="sub">años aproximados</div></div>
        </div>

        <div class="grid2">
          <div class="card">
            <h2>Diagnóstico cuantitativo</h2>
            <p class="lead">No se limita a decir “sí” o “no”: calcula cuánto margen tiene el proyecto antes de dejar de ser viable.</p>
            <div class="grid2" style="margin-top:0">
              <div class="mini-stat"><small>Caída de ingresos tolerable</small><strong id="diagRevenue">—</strong></div>
              <div class="mini-stat"><small>Aumento de gastos tolerable</small><strong id="diagExpense">—</strong></div>
              <div class="mini-stat"><small>Tasa bancaria de equilibrio</small><strong id="diagLoanRate">—</strong></div>
              <div class="mini-stat"><small>Inversión máxima soportada</small><strong id="diagMaxInv">—</strong></div>
            </div>
            <div id="mainCallout" class="callout good" style="margin-top:15px"></div>
          </div>

          <div class="card">
            <h2>Flujos ajustados y valor presente</h2>
            <p class="lead">Comparación anual del flujo después del costo financiero y su valor descontado.</p>
            <div class="bar-chart" id="mainBars"></div>
            <div class="legend"><span><i style="background:#16b9d4"></i>Flujo ajustado</span><span><i style="background:#2757b9"></i>Valor presente</span></div>
          </div>
        </div>

        <div class="card" style="margin-top:18px">
          <h2>Recomendación automática basada en números</h2>
          <p class="lead">El motor usa únicamente las variables del proyecto y análisis de punto de equilibrio; no inventa supuestos externos.</p>
          <div class="recommendations" id="recommendations"></div>
        </div>
      </section>

      <section class="section" id="proyecto">
        <div class="section-title"><div><h1>1. Definir el proyecto</h1><p>Inversión inicial, horizonte, tasa requerida e ingresos/gastos estimados.</p></div><span class="badge blue">Entradas editables</span></div>
        <div class="card">
          <div class="form-grid">
            <div class="field"><label>Nombre del proyecto</label><input id="projectName" value="Proyecto demostrativo · VAN con financiamiento"></div>
            <div class="field"><label>Moneda</label><select id="currency"><option value="CRC">₡ Colón</option><option value="USD">$ Dólar</option><option value="EUR">€ Euro</option></select></div>
            <div class="field"><label>Inversión inicial</label><input id="investment" type="number" min="0" step="1000" value="5600000"></div>
            <div class="field"><label>Tasa de descuento / costo de fondos (%)</label><input id="discountRate" type="number" min="0" max="200" step="0.1" value="20"></div>
            <div class="field"><label>Horizonte del proyecto (años)</label><input id="projectYears" type="number" min="1" max="10" step="1" value="4"></div>
            <div class="field"><label>Nombre del analista / equipo</label><input id="teamName" value="Equipo EC-300"></div>
          </div>
        </div>

        <div class="card" style="margin-top:18px">
          <div class="section-title" style="margin-bottom:14px"><div><h2>Proyección operativa</h2><p>Flujo Neto = Ingresos − Gastos.</p></div><span class="badge blue" id="flowYearsBadge">4 años</span></div>
          <div class="table-wrap">
            <table><thead><tr><th>Año</th><th>Ingresos</th><th>Gastos</th><th>Flujo neto</th><th>Margen</th></tr></thead><tbody id="operatingRows"></tbody></table>
          </div>
          <div class="callout" style="margin-top:14px"><b>Idea para la exposición:</b> justifique cómo estimó los ingresos y los gastos. El modelo financiero no sustituye esa justificación; la hace visible y comprobable.</div>
        </div>
      </section>

      <section class="section" id="credito">
        <div class="section-title"><div><h1>2. Diseñar el financiamiento</h1><p>Cuota nivelada (sistema francés), interés y amortización de capital.</p></div><span class="badge blue">Banco</span></div>
        <div class="grid2">
          <div class="card">
            <div class="form-grid" style="grid-template-columns:1fr 1fr">
              <div class="field"><label>Porcentaje financiado (%)</label><div class="inline-range"><input id="financePctRange" type="range" min="0" max="100" step="1" value="60"><input id="financePct" type="number" min="0" max="100" step="1" value="60"></div></div>
              <div class="field"><label>Tasa nominal anual del crédito (%)</label><input id="loanRate" type="number" min="0" max="200" step="0.1" value="14"></div>
              <div class="field"><label>Plazo del crédito (años)</label><input id="loanYears" type="number" min="1" max="30" step="1" value="4"></div>
              <div class="field"><label>Pagos por año</label><select id="paymentsPerYear"><option value="12" selected>12 · mensual</option><option value="4">4 · trimestral</option><option value="2">2 · semestral</option><option value="1">1 · anual</option></select></div>
            </div>
            <div id="termWarning" class="warning-strip"></div>
            <div class="callout" style="margin-top:16px">La tasa periódica se obtiene dividiendo la tasa nominal anual entre la periodicidad. En el ejemplo base: 14% / 12 = <b>1,1667% mensual</b>.</div>
          </div>
          <div class="card" style="display:grid;grid-template-columns:190px 1fr;gap:18px;align-items:center">
            <div class="finance-ring" id="financeRing"><div><strong id="ringPct">60%</strong><small>financiado</small></div></div>
            <div>
              <div class="mini-stat"><small>Monto financiado</small><strong id="loanAmount">—</strong></div>
              <div class="mini-stat" style="margin-top:9px"><small>Aporte propio</small><strong id="equityAmount">—</strong></div>
              <div class="mini-stat" style="margin-top:9px"><small>Cuota por período</small><strong id="periodPayment">—</strong></div>
            </div>
          </div>
        </div>

        <div class="kpis">
          <div class="kpi"><small>Servicio anual de deuda</small><div class="val" id="annualDebt">—</div><div class="sub">cuotas agregadas del año 1</div></div>
          <div class="kpi"><small>Intereses totales</small><div class="val" id="totalInterest">—</div><div class="sub">durante todo el crédito</div></div>
          <div class="kpi"><small>Capital amortizado año 1</small><div class="val" id="principalY1">—</div><div class="sub">reducción del pasivo</div></div>
          <div class="kpi"><small>Interés año 1</small><div class="val" id="interestY1">—</div><div class="sub">costo financiero</div></div>
        </div>

        <div class="card" style="margin-top:18px">
          <h2>Tabla anual de amortización</h2><p class="lead">La app calcula primero cada cuota periódica y luego consolida interés y capital por año, tal como exige el ejercicio.</p>
          <div class="table-wrap"><table><thead><tr><th>Año</th><th>Saldo inicial</th><th>Cuotas pagadas</th><th>Interés</th><th>Capital</th><th>Saldo final</th></tr></thead><tbody id="debtRows"></tbody></table></div>
        </div>
      </section>

      <section class="section" id="van">
        <div class="section-title"><div><h1>3. Calcular el VAN con financiamiento</h1><p>Descuento de los flujos ajustados al costo de fondos elegido.</p></div><span class="badge blue">Metodología de clase</span></div>
        <div class="grid2">
          <div class="card">
            <h2>Estructura matemática</h2>
            <div class="formula">
Flujo neto_t = Ingresos_t − Gastos_t

Monto financiado = Inversión × % financiamiento

Cuota periódica = P × i / [1 − (1+i)^(-n)]

Flujo de caja_t = Flujo neto_t − Cuotas_t

Flujo ajustado_t = Flujo de caja_t + Capital_t
                  = Flujo neto_t − Interés_t

VP_t = Flujo ajustado_t / (1+k)^t

VAN = − Inversión inicial + Σ VP_t
            </div>
          </div>
          <div class="card">
            <h2>Resultado</h2>
            <div class="mini-stat"><small>VAN</small><strong id="vanBig">—</strong></div>
            <div class="mini-stat" style="margin-top:10px"><small>TIR del flujo ajustado</small><strong id="irrBig">—</strong></div>
            <div class="mini-stat" style="margin-top:10px"><small>Spread TIR − costo de fondos</small><strong id="spreadBig">—</strong></div>
            <div id="vanVerdict" class="callout" style="margin-top:12px"></div>
          </div>
        </div>
        <div class="card" style="margin-top:18px">
          <h2>Auditoría completa del cálculo</h2><p class="lead">Todos los componentes quedan visibles para que el procedimiento pueda explicarse durante la presentación.</p>
          <div class="table-wrap"><table><thead><tr><th>Año</th><th>Ingresos</th><th>Gastos</th><th>Flujo neto</th><th>Cuotas</th><th>Capital</th><th>Interés</th><th>Flujo ajustado</th><th>Factor VP</th><th>Valor presente</th></tr></thead><tbody id="vanRows"></tbody></table></div>
        </div>
      </section>

      <section class="section" id="sensibilidad">
        <div class="section-title"><div><h1>4. Sensibilidad y “mesa de dibujo”</h1><p>¿Qué tendría que cambiar para fortalecer o rescatar el proyecto?</p></div><span class="badge blue">Buscar objetivo</span></div>
        <div class="kpis">
          <div class="kpi"><small>Ingreso de equilibrio</small><div class="val" id="beRevenue">—</div><div class="sub" id="beRevenueSub">escala uniforme</div></div>
          <div class="kpi"><small>Gasto de equilibrio</small><div class="val" id="beExpense">—</div><div class="sub" id="beExpenseSub">escala uniforme</div></div>
          <div class="kpi"><small>Tasa bancaria máxima</small><div class="val" id="beLoanRate">—</div><div class="sub">manteniendo lo demás constante</div></div>
          <div class="kpi"><small>Tasa de descuento máxima</small><div class="val" id="beDiscount">—</div><div class="sub">equivale a la TIR</div></div>
        </div>
        <div class="grid2">
          <div class="card">
            <h2>Escenario interactivo</h2><p class="lead">Pruebe shocks simultáneos sin modificar el escenario base.</p>
            <div class="field"><label>Variación de ingresos: <span id="revShockLabel">0%</span></label><input id="revShock" type="range" min="-30" max="30" step="1" value="0"></div>
            <div class="field" style="margin-top:14px"><label>Variación de gastos: <span id="expShockLabel">0%</span></label><input id="expShock" type="range" min="-30" max="30" step="1" value="0"></div>
            <div class="mini-stat" style="margin-top:18px"><small>VAN del escenario</small><strong id="scenarioNPV">—</strong></div>
            <div id="scenarioCallout" class="callout" style="margin-top:12px"></div>
          </div>
          <div class="card">
            <h2>Matriz de sensibilidad</h2><p class="lead">Columnas: variación de ingresos. Filas: variación de gastos.</p>
            <div style="overflow:auto"><div class="sens-grid" id="sensitivityGrid"></div></div>
          </div>
        </div>
        <div class="card" style="margin-top:18px">
          <h2>Plan de mejora cuantificado</h2>
          <div class="recommendations" id="improvementPlan"></div>
        </div>
      </section>

      <section class="section" id="exposicion">
        <div class="section-title"><div><h1>5. Modo exposición</h1><p>Estructura lista para explicar el proyecto de forma clara: introducción, datos, cálculo y conclusiones.</p></div><span class="badge blue">Presentación</span></div>
        <div class="presentation" id="presentation">
          <div class="slide active">
            <div class="kicker">1 · Introducción</div>
            <h2 id="slideTitle1">Evaluación de proyecto con VAN y financiamiento</h2>
            <p id="slideText1"></p>
          </div>
          <div class="slide">
            <div class="kicker">2 · Datos del proyecto</div>
            <h2>¿Qué estamos evaluando?</h2>
            <ul id="slideData"></ul>
          </div>
          <div class="slide">
            <div class="kicker">3 · Financiamiento bancario</div>
            <h2>La cuota se separa en <span class="accent">interés + capital</span></h2>
            <p id="slideDebt"></p>
          </div>
          <div class="slide">
            <div class="kicker">4 · Resultado</div>
            <h2>Valor Actual Neto</h2>
            <div class="big" id="slideNPV">—</div>
            <p id="slideResult"></p>
          </div>
          <div class="slide">
            <div class="kicker">5 · Conclusiones</div>
            <h2 id="slideConclusionTitle">Conclusión</h2>
            <ul id="slideConclusions"></ul>
          </div>
          <div class="slide-controls"><button id="prevSlide">← Anterior</button><span class="slide-counter" id="slideCounter">1 / 5</span><button id="nextSlide">Siguiente →</button></div>
        </div>
      </section>

      <div class="footer">
        FINOVA es un prototipo académico de apoyo a la evaluación de proyectos. Los resultados dependen de la calidad de los supuestos ingresados. Para decisiones reales deben validarse ingresos, gastos, tasas, impuestos, vida útil, valor residual y otros factores relevantes.
      </div>
    </div>
  </main>
</div>

<input id="importFile" type="file" accept=".json" style="display:none">

<script>
const DEFAULT = {
  projectName:"Proyecto demostrativo · VAN con financiamiento",
  teamName:"Equipo EC-300",
  currency:"CRC",
  investment:5600000,
  discountRate:20,
  projectYears:4,
  financePct:60,
  loanRate:14,
  loanYears:4,
  paymentsPerYear:12,
  incomes:[3744000,4212000,3510000,2340000],
  expenses:[672000,1572000,1022000,772000]
};

const $ = id => document.getElementById(id);
const clamp=(x,a,b)=>Math.min(Math.max(x,a),b);
const num=v=>Number(v)||0;
const pct=v=>`${Number(v).toFixed(2)}%`;
const symbols={CRC:"₡",USD:"$",EUR:"€"};
let slideIndex=0;

function money(v, compact=false){
  const cur=$("currency").value || "CRC";
  const opts={maximumFractionDigits:0,minimumFractionDigits:0};
  if(compact && Math.abs(v)>=1e6) return `${symbols[cur]}${(v/1e6).toFixed(2)} M`;
  return `${v<0?"−":""}${symbols[cur]}${Math.abs(v).toLocaleString("es-CR",opts)}`;
}
function signedPct(v){return `${v>=0?"+":""}${v.toFixed(2)}%`}

function showSection(id){
  document.querySelectorAll(".section").forEach(s=>s.classList.toggle("active",s.id===id));
  document.querySelectorAll(".nav button").forEach(b=>b.classList.toggle("active",b.dataset.section===id));
  window.scrollTo({top:0,behavior:"smooth"});
}
document.querySelectorAll(".nav button").forEach(b=>b.addEventListener("click",()=>showSection(b.dataset.section)));

function renderOperatingRows(preserve=true){
  const years=clamp(parseInt($("projectYears").value)||1,1,10);
  $("projectYears").value=years;
  const old=[];
  document.querySelectorAll("#operatingRows tr").forEach(tr=>{
    old.push({
      income:num(tr.querySelector(".income")?.value),
      expense:num(tr.querySelector(".expense")?.value)
    });
  });
  $("operatingRows").innerHTML="";
  for(let i=0;i<years;i++){
    const d=preserve && old[i] ? old[i] : {income:DEFAULT.incomes[i]||0,expense:DEFAULT.expenses[i]||0};
    const tr=document.createElement("tr");
    tr.innerHTML=`<td><b>Año ${i+1}</b></td>
      <td><input class="row-input income" type="number" min="0" step="1000" value="${Math.round(d.income)}"></td>
      <td><input class="row-input expense" type="number" min="0" step="1000" value="${Math.round(d.expense)}"></td>
      <td class="net">—</td><td class="margin">—</td>`;
    $("operatingRows").appendChild(tr);
  }
  $("flowYearsBadge").textContent=`${years} ${years===1?"año":"años"}`;
  document.querySelectorAll("#operatingRows input").forEach(i=>i.addEventListener("input",update));
}

function readInputs(){
  const rows=[...document.querySelectorAll("#operatingRows tr")];
  return {
    projectName:$("projectName").value.trim()||"Proyecto sin nombre",
    teamName:$("teamName").value.trim()||"Equipo",
    currency:$("currency").value,
    investment:Math.max(0,num($("investment").value)),
    discountRate:Math.max(0,num($("discountRate").value))/100,
    years:rows.length,
    financePct:clamp(num($("financePct").value),0,100)/100,
    loanRate:Math.max(0,num($("loanRate").value))/100,
    loanYears:Math.max(1,parseInt($("loanYears").value)||1),
    ppy:Math.max(1,parseInt($("paymentsPerYear").value)||12),
    incomes:rows.map(r=>Math.max(0,num(r.querySelector(".income").value))),
    expenses:rows.map(r=>Math.max(0,num(r.querySelector(".expense").value)))
  };
}

function payment(principal, annualRate, ppy, loanYears){
  const n=Math.round(ppy*loanYears);
  if(principal<=0||n<=0)return 0;
  const i=annualRate/ppy;
  return Math.abs(i)<1e-12 ? principal/n : principal*i/(1-Math.pow(1+i,-n));
}

function buildDebtSchedule(d, overrideLoanRate=null){
  const rate=overrideLoanRate===null?d.loanRate:overrideLoanRate;
  const principal=d.investment*d.financePct;
  const n=Math.round(d.loanYears*d.ppy);
  const per=payment(principal,rate,d.ppy,d.loanYears);
  let bal=principal;
  const years=[];
  let totalInterest=0,totalPaid=0;
  for(let m=0;m<n;m++){
    const yr=Math.floor(m/d.ppy);
    if(!years[yr]) years[yr]={year:yr+1,opening:bal,payment:0,interest:0,principal:0,closing:bal};
    const i=rate/d.ppy;
    const interest=bal*i;
    let principalPaid=Math.max(0,per-interest);
    let actualPayment=per;
    if(m===n-1 || principalPaid>bal){principalPaid=bal;actualPayment=bal+interest}
    bal=Math.max(0,bal-principalPaid);
    years[yr].payment+=actualPayment;
    years[yr].interest+=interest;
    years[yr].principal+=principalPaid;
    years[yr].closing=bal;
    totalInterest+=interest; totalPaid+=actualPayment;
  }
  return {principal,per,years,totalInterest,totalPaid,balance:bal};
}

function npv(rate, flows){
  return flows.reduce((acc,x,i)=>acc+x/Math.pow(1+rate,i),0);
}
function irr(flows){
  let low=-0.9999, high=10;
  let fLow=npv(low,flows), fHigh=npv(high,flows);
  if(!isFinite(fLow)||!isFinite(fHigh)||fLow*fHigh>0) return null;
  for(let k=0;k<180;k++){
    const mid=(low+high)/2, f=npv(mid,flows);
    if(Math.abs(f)<1e-8)return mid;
    if(fLow*f<=0){high=mid;fHigh=f}else{low=mid;fLow=f}
  }
  return (low+high)/2;
}
function discountedPayback(investment,pvs){
  let cum=0;
  for(let i=0;i<pvs.length;i++){
    if(cum+pvs[i]>=investment){
      const rem=investment-cum;
      return i+(pvs[i]!==0?rem/pvs[i]:1);
    }
    cum+=pvs[i];
  }
  return null;
}
function simplePayback(investment,flows){
  let cum=0;
  for(let i=0;i<flows.length;i++){
    if(cum+flows[i]>=investment){
      const rem=investment-cum;
      return i+(flows[i]!==0?rem/flows[i]:1);
    }
    cum+=flows[i];
  }
  return null;
}

function model(d, scenario={revShock:0,expShock:0,loanRate:null}){
  const ds=buildDebtSchedule(d,scenario.loanRate);
  const incomes=d.incomes.map(x=>x*(1+scenario.revShock));
  const expenses=d.expenses.map(x=>x*(1+scenario.expShock));
  const rows=[];
  for(let i=0;i<d.years;i++){
    const debt=ds.years[i]||{payment:0,interest:0,principal:0,opening:0,closing:0};
    const net=incomes[i]-expenses[i];
    const cash=net-debt.payment;
    const adjusted=cash+debt.principal; // teacher/class method: net - interest
    const factor=1/Math.pow(1+d.discountRate,i+1);
    const pv=adjusted*factor;
    rows.push({year:i+1,income:incomes[i],expense:expenses[i],net,debtPayment:debt.payment,interest:debt.interest,principal:debt.principal,cash,adjusted,factor,pv});
  }
  const van=-d.investment+rows.reduce((a,r)=>a+r.pv,0);
  const flows=[-d.investment,...rows.map(r=>r.adjusted)];
  const projectIRR=irr(flows);
  const dpb=discountedPayback(d.investment,rows.map(r=>r.pv));
  const spb=simplePayback(d.investment,rows.map(r=>r.adjusted));
  return {d,ds,rows,van,projectIRR,dpb,spb,flows};
}

function breakEvens(m){
  const d=m.d, r=m.rows;
  const pvRevenue=r.reduce((a,x)=>a+x.income*x.factor,0);
  const pvExpense=r.reduce((a,x)=>a+x.expense*x.factor,0);
  const pvInterest=r.reduce((a,x)=>a+x.interest*x.factor,0);
  const revFactor=pvRevenue>0?(d.investment+pvExpense+pvInterest)/pvRevenue:null;
  const expFactor=pvExpense>0?(pvRevenue-pvInterest-d.investment)/pvExpense:null;
  const maxInvestment=r.reduce((a,x)=>a+x.pv,0);

  // Find bank rate where VAN = 0.
  let maxLoanRate=null;
  if(d.financePct===0){ maxLoanRate=Infinity; }
  else{
    const f = rate => model(d,{loanRate:rate}).van;
    let lo=0, hi=2.0, flo=f(lo), fhi=f(hi);
    if(flo<0) maxLoanRate=0;
    else if(fhi>0) maxLoanRate=Infinity;
    else{
      for(let k=0;k<100;k++){
        const mid=(lo+hi)/2, fm=f(mid);
        if(fm>=0)lo=mid;else hi=mid;
      }
      maxLoanRate=(lo+hi)/2;
    }
  }
  return {revFactor,expFactor,maxInvestment,maxLoanRate};
}

function status(m){
  const margin=m.d.investment?m.van/m.d.investment:0;
  if(m.van>1e-6){
    if(margin<.05) return {label:"SÍ · VIABLE",tone:"orange",text:"El VAN es positivo, aunque el margen sobre la inversión inicial es relativamente estrecho. Conviene revisar sensibilidad antes de comprometer el capital."};
    return {label:"SÍ · VIABLE",tone:"green",text:"El VAN es positivo: bajo los supuestos ingresados, el proyecto crea valor por encima de la tasa requerida."};
  }
  if(m.van<-1e-6) return {label:"NO · REPLANTEAR",tone:"red",text:"El VAN es negativo. Bajo estas condiciones el proyecto destruye valor; conviene regresar a la mesa de dibujo antes de invertir."};
  return {label:"INDIFERENTE",tone:"orange",text:"El VAN está prácticamente en cero: el proyecto apenas cubre el rendimiento requerido."};
}

function updateOperatingDisplay(m){
  const trs=[...document.querySelectorAll("#operatingRows tr")];
  trs.forEach((tr,i)=>{
    const r=m.rows[i], margin=r.income?100*r.net/r.income:0;
    tr.querySelector(".net").textContent=money(r.net);
    tr.querySelector(".margin").textContent=pct(margin);
  });
}

function renderBars(m){
  const el=$("mainBars");el.innerHTML="";
  const vals=m.rows.flatMap(r=>[Math.max(0,r.adjusted),Math.max(0,r.pv)]);
  const max=Math.max(...vals,1);
  m.rows.forEach(r=>{
    const c=document.createElement("div");c.className="bar-col";
    const h1=Math.max(4,170*Math.max(0,r.adjusted)/max),h2=Math.max(4,170*Math.max(0,r.pv)/max);
    c.innerHTML=`<div class="bar flow" title="${money(r.adjusted)}" style="height:${h1}px"></div><div class="bar pv" title="${money(r.pv)}" style="height:${h2}px"></div><label>Año ${r.year}</label>`;
    el.appendChild(c);
  });
}

function renderDebt(m){
  const ds=m.ds,d=m.d;
  $("financeRing").style.background=`conic-gradient(var(--cyan) 0 ${d.financePct*100}%,#dce5f2 ${d.financePct*100}% 100%)`;
  $("ringPct").textContent=`${(d.financePct*100).toFixed(0)}%`;
  $("loanAmount").textContent=money(ds.principal);
  $("equityAmount").textContent=money(d.investment-ds.principal);
  $("periodPayment").textContent=money(ds.per);
  $("annualDebt").textContent=money(ds.years[0]?.payment||0);
  $("totalInterest").textContent=money(ds.totalInterest);
  $("principalY1").textContent=money(ds.years[0]?.principal||0);
  $("interestY1").textContent=money(ds.years[0]?.interest||0);
  $("debtRows").innerHTML=ds.years.map(y=>`<tr><td><b>Año ${y.year}</b></td><td>${money(y.opening)}</td><td>${money(y.payment)}</td><td>${money(y.interest)}</td><td>${money(y.principal)}</td><td>${money(y.closing)}</td></tr>`).join("");
  const warning=$("termWarning");
  if(d.loanYears>d.years){
    warning.style.display="block";
    warning.innerHTML=`⚠ El crédito dura ${d.loanYears} años, pero el proyecto solo proyecta ${d.years}. Extienda el horizonte para incorporar todos los intereses futuros; de lo contrario, el VAN debe interpretarse como preliminar.`;
  }else warning.style.display="none";
}

function renderVAN(m){
  $("vanRows").innerHTML=m.rows.map(r=>`<tr><td><b>${r.year}</b></td><td>${money(r.income)}</td><td>${money(r.expense)}</td><td>${money(r.net)}</td><td>${money(r.debtPayment)}</td><td>${money(r.principal)}</td><td>${money(r.interest)}</td><td>${money(r.adjusted)}</td><td>${r.factor.toFixed(4)}</td><td><b>${money(r.pv)}</b></td></tr>`).join("");
  $("vanBig").textContent=money(m.van);
  $("irrBig").textContent=m.projectIRR===null?"No definida":pct(m.projectIRR*100);
  const spread=m.projectIRR===null?null:(m.projectIRR-m.d.discountRate)*100;
  $("spreadBig").textContent=spread===null?"—":`${spread>=0?"+":""}${spread.toFixed(2)} p.p.`;
  const st=status(m),v=$("vanVerdict");
  v.className=`callout ${st.tone==="green"?"good":st.tone==="red"?"bad":"warn"}`;
  v.innerHTML=`<b>${st.label}.</b> ${st.text}`;
}

function renderSensitivity(m,be){
  const revDelta=be.revFactor===null?null:(be.revFactor-1)*100;
  const expDelta=be.expFactor===null?null:(be.expFactor-1)*100;
  $("beRevenue").textContent=be.revFactor===null?"—":`${be.revFactor.toFixed(3)}×`;
  $("beRevenueSub").textContent=revDelta===null?"—":revDelta<0?`pueden caer ${Math.abs(revDelta).toFixed(2)}%`:`deben subir ${revDelta.toFixed(2)}%`;
  $("beExpense").textContent=be.expFactor===null?"—":`${be.expFactor.toFixed(3)}×`;
  $("beExpenseSub").textContent=expDelta===null?"—":expDelta>0?`pueden subir ${expDelta.toFixed(2)}%`:`deben bajar ${Math.abs(expDelta).toFixed(2)}%`;
  $("beLoanRate").textContent=be.maxLoanRate===Infinity?"Sin límite en rango":pct(be.maxLoanRate*100);
  $("beDiscount").textContent=m.projectIRR===null?"—":pct(m.projectIRR*100);

  const revs=[-10,-5,0,5,10], exps=[10,5,0,-5,-10];
  let html=`<div class="sens-cell sens-head">Gastos \\ Ingresos</div>`+revs.map(x=>`<div class="sens-cell sens-head">${signedPct(x)}</div>`).join("");
  exps.forEach(e=>{
    html+=`<div class="sens-cell sens-head">${signedPct(e)}</div>`;
    revs.forEach(r=>{
      const v=model(m.d,{revShock:r/100,expShock:e/100}).van;
      const cls=v>m.d.investment*.01?"sens-pos":v<-m.d.investment*.01?"sens-neg":"sens-mid";
      html+=`<div class="sens-cell ${cls}" title="${money(v)}">${money(v,true)}</div>`;
    });
  });
  $("sensitivityGrid").innerHTML=html;

  const rs=num($("revShock").value), es=num($("expShock").value);
  $("revShockLabel").textContent=signedPct(rs); $("expShockLabel").textContent=signedPct(es);
  const scen=model(m.d,{revShock:rs/100,expShock:es/100});
  $("scenarioNPV").textContent=money(scen.van);
  const sc=$("scenarioCallout");
  sc.className=`callout ${scen.van>=0?"good":"bad"}`;
  sc.innerHTML=scen.van>=0?`El escenario sigue siendo viable: mantiene un VAN de <b>${money(scen.van)}</b>.`:`El escenario deja de ser viable: el VAN cae a <b>${money(scen.van)}</b>.`;
}

function renderRecommendations(m,be){
  const revDelta=(be.revFactor-1)*100, expDelta=(be.expFactor-1)*100;
  const maxRate=be.maxLoanRate, currentRate=m.d.loanRate;
  const spread=m.projectIRR===null?null:(m.projectIRR-m.d.discountRate)*100;
  const headroomRate=maxRate===Infinity?null:(maxRate-currentRate)*100;
  const st=status(m);

  const recs=[];
  recs.push(`<b>Decisión:</b> ${st.label}. VAN = <b>${money(m.van)}</b>${m.projectIRR!==null?` y TIR ajustada = <b>${pct(m.projectIRR*100)}</b>`:""}.`);
  if(revDelta>0) recs.push(`Para llegar a VAN = 0 solo mediante ingresos, se requiere elevar la proyección de ingresos en aproximadamente <b>${revDelta.toFixed(2)}%</b>, manteniendo los demás supuestos constantes.`);
  else recs.push(`El proyecto puede absorber una caída uniforme de ingresos de aproximadamente <b>${Math.abs(revDelta).toFixed(2)}%</b> antes de que el VAN llegue a cero.`);
  if(expDelta<0) recs.push(`Para llegar a VAN = 0 solo mediante costos, los gastos tendrían que reducirse aproximadamente <b>${Math.abs(expDelta).toFixed(2)}%</b>.`);
  else recs.push(`Los gastos podrían aumentar alrededor de <b>${expDelta.toFixed(2)}%</b> antes de que el VAN llegue a cero.`);
  if(maxRate!==Infinity) recs.push(`Con el resto de variables fijas, la tasa nominal bancaria de equilibrio es cercana a <b>${pct(maxRate*100)}</b>${headroomRate!==null?`; el crédito actual tiene un colchón de <b>${headroomRate.toFixed(2)} p.p.</b>`:""}.`);
  if(spread!==null) recs.push(`La TIR supera la tasa requerida en <b>${spread.toFixed(2)} p.p.</b>. Ese spread es el margen de rendimiento frente al costo de fondos.`);

  $("recommendations").innerHTML=recs.slice(0,4).map((x,i)=>`<div class="rec"><div class="num">${i+1}</div><p>${x}</p></div>`).join("");

  const plan=[];
  if(m.van<0){
    plan.push(`Subir ingresos al menos <b>${Math.max(0,revDelta).toFixed(2)}%</b> si esa fuera la única palanca.`);
    plan.push(`O reducir gastos al menos <b>${Math.max(0,-expDelta).toFixed(2)}%</b> si esa fuera la única palanca.`);
    if(maxRate!==Infinity && maxRate<currentRate) plan.push(`Renegociar la tasa del crédito desde <b>${pct(currentRate*100)}</b> hacia <b>${pct(maxRate*100)}</b> o menos.`);
    plan.push(`Alternativamente, reducir la inversión inicial hasta aproximadamente <b>${money(be.maxInvestment)}</b>.`);
  } else {
    plan.push(`El proyecto crea valor, pero conviene conservar un margen: el VAN equivale a <b>${(100*m.van/m.d.investment).toFixed(2)}%</b> de la inversión inicial.`);
    plan.push(`La inversión total podría subir hasta aproximadamente <b>${money(be.maxInvestment)}</b> antes de que VAN = 0, si los flujos no cambian.`);
    plan.push(revDelta<0?`La proyección tolera una caída de ingresos de <b>${Math.abs(revDelta).toFixed(2)}%</b>. Una caída mayor exige rediseñar precio, volumen o estructura de costos.`:`Los ingresos deben subir <b>${revDelta.toFixed(2)}%</b> para alcanzar equilibrio.`);
    if(maxRate!==Infinity) plan.push(`Evite aceptar una tasa bancaria superior a aproximadamente <b>${pct(maxRate*100)}</b> bajo estos mismos flujos.`);
  }
  $("improvementPlan").innerHTML=plan.map((x,i)=>`<div class="rec"><div class="num">${i+1}</div><p>${x}</p></div>`).join("");
}

function renderPresentation(m,be){
  const d=m.d, st=status(m);
  $("slideTitle1").textContent=`${d.projectName}: evaluación con VAN y financiamiento`;
  $("slideText1").innerHTML=`El objetivo es determinar si una inversión inicial de <b>${money(d.investment)}</b>, financiada en <b>${(d.financePct*100).toFixed(0)}%</b> con crédito bancario, genera valor por encima de una tasa requerida de <b>${pct(d.discountRate*100)}</b>.`;
  $("slideData").innerHTML=[
    `Inversión inicial: <b>${money(d.investment)}</b>.`,
    `Horizonte: <b>${d.years} años</b>.`,
    `Financiamiento: <b>${(d.financePct*100).toFixed(0)}%</b> a una tasa nominal anual de <b>${pct(d.loanRate*100)}</b>.`,
    `Ingresos proyectados totales: <b>${money(d.incomes.reduce((a,b)=>a+b,0))}</b>.`,
    `Gastos proyectados totales: <b>${money(d.expenses.reduce((a,b)=>a+b,0))}</b>.`
  ].map(x=>`<li>${x}</li>`).join("");
  $("slideDebt").innerHTML=`El banco financia <b>${money(m.ds.principal)}</b>. La cuota periódica nivelada es <b>${money(m.ds.per)}</b>. En el primer año se pagan <b>${money(m.ds.years[0]?.interest||0)}</b> de interés y se amortizan <b>${money(m.ds.years[0]?.principal||0)}</b> de capital. Para el VAN de clase, el capital se agrega de nuevo porque reduce el pasivo y no es gasto.`;
  $("slideNPV").textContent=money(m.van);
  $("slideResult").innerHTML=`${st.text} ${m.projectIRR!==null?`La TIR del flujo ajustado es <b>${pct(m.projectIRR*100)}</b>, frente a una tasa requerida de <b>${pct(d.discountRate*100)}</b>.`:""}`;
  $("slideConclusionTitle").textContent=m.van>=0?"Conclusión: el proyecto es viable bajo estos supuestos":"Conclusión: regresar a la mesa de dibujo";
  const revDelta=(be.revFactor-1)*100, expDelta=(be.expFactor-1)*100;
  $("slideConclusions").innerHTML=[
    `VAN = <b>${money(m.van)}</b>: ${m.van>=0?"crea valor":"no cubre el rendimiento requerido"}.`,
    m.projectIRR!==null?`TIR ajustada = <b>${pct(m.projectIRR*100)}</b>.`:`La TIR no está definida para estos flujos.`,
    revDelta<0?`El proyecto tolera una caída de ingresos de aproximadamente <b>${Math.abs(revDelta).toFixed(2)}%</b>.`:`Para equilibrarse, los ingresos deben subir aproximadamente <b>${revDelta.toFixed(2)}%</b>.`,
    expDelta>0?`Los gastos pueden subir alrededor de <b>${expDelta.toFixed(2)}%</b> antes de VAN = 0.`:`Los gastos deben bajar alrededor de <b>${Math.abs(expDelta).toFixed(2)}%</b>.`
  ].map(x=>`<li>${x}</li>`).join("");
}

function update(){
  $("financePctRange").value=$("financePct").value;
  const d=readInputs(),m=model(d),be=breakEvens(m),st=status(m);
  $("topProject").textContent=d.projectName;$("heroProject").textContent=d.projectName;
  $("decisionMain").textContent=st.label;$("decisionSub").textContent=st.text;
  $("kpiNPV").textContent=money(m.van);
  $("kpiNPVSub").textContent=`${m.van>=0?"+":""}${(100*m.van/(d.investment||1)).toFixed(2)}% de la inversión`;
  $("kpiIRR").textContent=m.projectIRR===null?"—":pct(m.projectIRR*100);
  $("kpiIRRSub").textContent=m.projectIRR===null?"No definida":`Tasa requerida ${pct(d.discountRate*100)}`;
  $("kpiDebt").textContent=money(m.ds.principal);
  $("kpiDebtSub").textContent=`${(d.financePct*100).toFixed(0)}% financiado`;
  $("kpiPayback").textContent=m.dpb===null?"> horizonte":m.dpb.toFixed(2);

  const revDelta=(be.revFactor-1)*100,expDelta=(be.expFactor-1)*100;
  $("diagRevenue").textContent=revDelta<0?`−${Math.abs(revDelta).toFixed(2)}%`:`+${revDelta.toFixed(2)}%`;
  $("diagExpense").textContent=expDelta>0?`+${expDelta.toFixed(2)}%`:`−${Math.abs(expDelta).toFixed(2)}%`;
  $("diagLoanRate").textContent=be.maxLoanRate===Infinity?"—":pct(be.maxLoanRate*100);
  $("diagMaxInv").textContent=money(be.maxInvestment);
  const mc=$("mainCallout");mc.className=`callout ${m.van>=0?"good":"bad"}`;
  mc.innerHTML=m.van>=0?`<b>Lectura:</b> el proyecto supera el costo de fondos. El margen no debe interpretarse como garantía: depende de que los ingresos y gastos proyectados sean realistas.`:`<b>Lectura:</b> el proyecto no alcanza el rendimiento requerido. Use la sección de sensibilidad para cuantificar cuánto deben cambiar las variables.`;

  updateOperatingDisplay(m);renderBars(m);renderDebt(m);renderVAN(m);renderSensitivity(m,be);renderRecommendations(m,be);renderPresentation(m,be);
  saveLocal();
}

function loadDefault(){
  $("projectName").value=DEFAULT.projectName;$("teamName").value=DEFAULT.teamName;$("currency").value=DEFAULT.currency;
  $("investment").value=DEFAULT.investment;$("discountRate").value=DEFAULT.discountRate;$("projectYears").value=DEFAULT.projectYears;
  $("financePct").value=DEFAULT.financePct;$("financePctRange").value=DEFAULT.financePct;$("loanRate").value=DEFAULT.loanRate;
  $("loanYears").value=DEFAULT.loanYears;$("paymentsPerYear").value=DEFAULT.paymentsPerYear;
  renderOperatingRows(false);
  update();
}
function saveLocal(){
  try{
    const d=readInputs();
    localStorage.setItem("finova-ec300",JSON.stringify({
      ...d,discountRate:d.discountRate*100,financePct:d.financePct*100,loanRate:d.loanRate*100,projectYears:d.years
    }));
  }catch(e){}
}
function exportScenario(){
  const d=readInputs();
  const payload={version:"FINOVA-EC300-1",...d,discountRate:d.discountRate*100,financePct:d.financePct*100,loanRate:d.loanRate*100,projectYears:d.years};
  const blob=new Blob([JSON.stringify(payload,null,2)],{type:"application/json"});
  const a=document.createElement("a");a.href=URL.createObjectURL(blob);a.download="escenario_VAN_financiamiento.json";a.click();URL.revokeObjectURL(a.href);
}

$("btnExample").addEventListener("click",loadDefault);
$("btnExport").addEventListener("click",exportScenario);
$("financePctRange").addEventListener("input",()=>{$("financePct").value=$("financePctRange").value;update()});
$("financePct").addEventListener("input",()=>{$("financePctRange").value=$("financePct").value;update()});
$("projectYears").addEventListener("change",()=>{renderOperatingRows(true);update()});
["projectName","teamName","currency","investment","discountRate","loanRate","loanYears","paymentsPerYear","financePct","revShock","expShock"].forEach(id=>{
  $(id).addEventListener("input",update);$(id).addEventListener("change",update);
});

function setSlide(n){
  const slides=[...document.querySelectorAll(".slide")];
  slideIndex=(n+slides.length)%slides.length;
  slides.forEach((s,i)=>s.classList.toggle("active",i===slideIndex));
  $("slideCounter").textContent=`${slideIndex+1} / ${slides.length}`;
}
$("prevSlide").addEventListener("click",()=>setSlide(slideIndex-1));
$("nextSlide").addEventListener("click",()=>setSlide(slideIndex+1));
document.addEventListener("keydown",e=>{
  if(document.querySelector("#exposicion.section.active")){
    if(e.key==="ArrowRight")setSlide(slideIndex+1);
    if(e.key==="ArrowLeft")setSlide(slideIndex-1);
  }
});

loadDefault();
</script>
</body>
</html>
"""

output = Path("FINOVA_VAN_Financiamiento_EC300.html")
output.write_text(HTML_APP, encoding="utf-8")

display(Markdown(f"### ✅ Aplicación generada\nArchivo: `{output.resolve()}`"))
display(IFrame(src=str(output), width="100%", height=900))


### ✅ Aplicación generada
Archivo: `/Users/juani/Documents/GitHub/FINOVA/FINOVA_VAN_Financiamiento_EC300.html`